
# iNaturalist Coarse-Start Real-Trajectory Pilot

This notebook replaces the first real-data pilot, which mostly sampled observations
that were already at species level almost immediately.

## What changes

Instead of sampling generic research-grade observations, this notebook searches the
**identification stream** for observations that contain a coarse identification
(genus/family/order/class) while the current observation taxon is species-level.
It then retrieves the complete observation history and keeps only observations whose
**first compatible identification is genuinely coarse**.

Time zero is the first compatible identification, not observation creation.

The model uses horizons:

\[
t\in\{0,1,7,30,90\}\text{ days after the first compatible ID.}
\]

The statistical criterion also follows the corrected **sample-analogue** interpretation:
the eventual hidden species proportions are reported retrospectively, but they are
**not required** to lie inside the finite-sample empirical sensitivity polytope.

No images are downloaded and no GPU is required.


In [ ]:

import json, math, time, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.optimize import linprog

API = "https://api.inaturalist.org/v1"
HEADERS = {
    "User-Agent": "SelectiveRefinementCoarseStartPilot/1.0 (public metadata research)"
}

TARGET_TAXON_QUERY = "Aves"
TARGET_TAXON_RANK = "class"

OBS_CREATED_D1 = "2022-01-01"
OBS_CREATED_D2 = "2024-12-31"

COARSE_RANKS = ["genus", "family", "order", "class"]
TARGET_CANDIDATE_OBSERVATIONS = 4000
MAX_ID_PAGES_PER_RANK = 20
PER_PAGE = 200

HORIZON_DAYS = [0, 1, 7, 30, 90]

MAX_GENERA = 8
MAX_SPECIES_PER_GENUS = 5
MIN_OBS_PER_SPECIES = 15
MIN_SPECIES_PER_GENUS = 2

GAMMA_GRID = [2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 20.0]

REQUEST_SLEEP = 0.25
MAX_RETRIES = 5
REQUEST_TIMEOUT = 60

OUTDIR = Path("inaturalist_coarse_start_trajectory_pilot_results")
OUTDIR.mkdir(exist_ok=True)

print("Coarse-start real trajectory pilot")
print("Observation-created window:", OBS_CREATED_D1, "to", OBS_CREATED_D2)
print("Coarse ranks:", COARSE_RANKS)


## 1. API helpers and target taxon

In [ ]:

session = requests.Session()
session.headers.update(HEADERS)

def api_get(path, params=None, timeout=REQUEST_TIMEOUT):
    url = f"{API}/{path.lstrip('/')}"
    last = None

    for attempt in range(MAX_RETRIES):
        try:
            r = session.get(url, params=params, timeout=timeout)

            if r.status_code == 429:
                time.sleep(min(30, 2 ** attempt))
                continue

            r.raise_for_status()
            return r.json()

        except Exception as e:
            last = e
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(min(15, 2 ** attempt))

    raise RuntimeError(last)

def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

taxon_search = api_get(
    "taxa/autocomplete",
    params={
        "q": TARGET_TAXON_QUERY,
        "rank": TARGET_TAXON_RANK,
        "per_page": 10,
    },
)

if not taxon_search.get("results"):
    raise RuntimeError("Could not resolve target taxon")

target_taxon = taxon_search["results"][0]
TARGET_TAXON_ID = int(target_taxon["id"])

print({
    "id": TARGET_TAXON_ID,
    "name": target_taxon.get("name"),
    "rank": target_taxon.get("rank"),
})



## 2. Find observations with coarse identifications

The iNaturalist identification search supports filters on the identification's rank
and on the current observation's rank/taxon. We use it only as a **candidate generator**.
Complete histories are fetched later and independently re-audited.


In [ ]:

def observation_id_from_identification(ident):
    obs = ident.get("observation")

    if isinstance(obs, dict) and obs.get("id") is not None:
        return int(obs["id"])

    if ident.get("observation_id") is not None:
        return int(ident["observation_id"])

    return None

candidate_obs_ids = set()
candidate_id_rows = []

for rank in COARSE_RANKS:
    for page in range(1, MAX_ID_PAGES_PER_RANK + 1):
        params = {
            "taxon_id": TARGET_TAXON_ID,
            "rank": rank,
            "observation_taxon_id": TARGET_TAXON_ID,
            "observation_rank": "species",
            "observation_created_d1": OBS_CREATED_D1,
            "observation_created_d2": OBS_CREATED_D2,
            "per_page": PER_PAGE,
            "page": page,
            "order_by": "id",
            "order": "asc",
        }

        data = api_get("identifications", params=params)
        batch = data.get("results", [])

        if not batch:
            break

        for ident in batch:
            oid = observation_id_from_identification(ident)

            if oid is not None:
                candidate_obs_ids.add(oid)
                candidate_id_rows.append({
                    "observation_id": oid,
                    "candidate_identification_id": ident.get("id"),
                    "candidate_rank": rank,
                    "candidate_taxon_id": (ident.get("taxon") or {}).get("id"),
                    "candidate_created_at": ident.get("created_at"),
                })

        print(
            f"rank={rank:7s} page={page:2d}: "
            f"{len(batch)} IDs, {len(candidate_obs_ids)} unique observations"
        )

        time.sleep(REQUEST_SLEEP)

        if len(candidate_obs_ids) >= TARGET_CANDIDATE_OBSERVATIONS:
            break

        if len(batch) < PER_PAGE:
            break

    if len(candidate_obs_ids) >= TARGET_CANDIDATE_OBSERVATIONS:
        break

candidate_obs_ids = sorted(candidate_obs_ids)[:TARGET_CANDIDATE_OBSERVATIONS]

candidate_ids_df = pd.DataFrame(candidate_id_rows)
candidate_ids_df.to_csv(
    OUTDIR/"coarse_identification_candidate_hits.csv",
    index=False
)

if len(candidate_obs_ids) < 500:
    raise RuntimeError(
        f"Only {len(candidate_obs_ids)} coarse-ID candidate observations found. "
        "Broaden the date window or increase pages."
    )

print("Candidate observations:", len(candidate_obs_ids))


## 3. Fetch complete observation histories safely

In [ ]:

def fetch_observation_details(ids, batch_size=50):
    full = []

    def fetch_batch(batch):
        if not batch:
            return []

        path = "observations/" + ",".join(str(x) for x in batch)

        try:
            return api_get(path).get("results", [])

        except requests.HTTPError as e:
            status = getattr(e.response, "status_code", None)

            if status in (414, 422) and len(batch) > 1:
                mid = len(batch) // 2
                return (
                    fetch_batch(batch[:mid])
                    + fetch_batch(batch[mid:])
                )
            raise

    batches = list(chunks(ids, batch_size))

    for bi, batch in enumerate(batches, start=1):
        got = fetch_batch(batch)
        full.extend(got)

        print(
            f"detail {bi}/{len(batches)}: "
            f"requested={len(batch)}, returned={len(got)}, total={len(full)}"
        )
        time.sleep(REQUEST_SLEEP)

    dedup = {
        int(o["id"]): o
        for o in full
        if o.get("id") is not None
    }

    return list(dedup.values())

observations = fetch_observation_details(candidate_obs_ids, 50)

print("Full observation histories:", len(observations))


## 4. Taxonomy metadata

In [ ]:

all_taxon_ids = {TARGET_TAXON_ID}

for o in observations:
    tax = o.get("taxon") or {}

    if tax.get("id") is not None:
        all_taxon_ids.add(int(tax["id"]))

    for aid in tax.get("ancestor_ids", []) or []:
        all_taxon_ids.add(int(aid))

def fetch_taxa(ids):
    out = {}

    for batch in chunks(sorted(ids), 30):
        data = api_get(
            "taxa/" + ",".join(str(x) for x in batch)
        )

        for t in data.get("results", []):
            out[int(t["id"])] = t

        time.sleep(REQUEST_SLEEP)

    return out

taxa = fetch_taxa(all_taxon_ids)

def final_lineage(o):
    tax = o["taxon"]
    return (
        [int(x) for x in (tax.get("ancestor_ids") or [])]
        + [int(tax["id"])]
    )

def rank_ancestor_id(o, rank):
    lin = final_lineage(o)

    vals = [
        tid for tid in lin
        if taxa.get(tid, {}).get("rank") == rank
    ]

    return vals[-1] if vals else None

print("Taxa fetched:", len(taxa))


## 5. Keep genuinely coarse-start, research-grade species observations

In [ ]:

def parse_time(x):
    return pd.to_datetime(x, utc=True, errors="coerce")

audit_rows = []
eligible = []

for o in observations:
    tax = o.get("taxon") or {}

    if (
        o.get("quality_grade") != "research"
        or tax.get("rank") != "species"
        or tax.get("id") is None
    ):
        continue

    sid = int(tax["id"])
    gid = rank_ancestor_id(o, "genus")

    if gid is None:
        continue

    lin = final_lineage(o)
    linset = set(lin)

    ids = sorted(
        list(o.get("identifications") or []),
        key=lambda z: parse_time(z.get("created_at"))
    )

    compatible = []

    for ident in ids:
        itax = ident.get("taxon") or {}

        if itax.get("id") is None:
            continue

        tid = int(itax["id"])
        t = parse_time(ident.get("created_at"))

        if pd.isna(t):
            continue

        if tid in linset:
            compatible.append((t, tid, itax.get("rank"), ident))

    if not compatible:
        continue

    first_t, first_tid, first_rank, first_ident = compatible[0]
    coarse_start = first_rank != "species"

    audit_rows.append({
        "observation_id": int(o["id"]),
        "species_id": sid,
        "species_name": taxa.get(sid, tax).get("name"),
        "genus_id": gid,
        "genus_name": taxa.get(gid, {}).get("name"),
        "first_compatible_rank": first_rank,
        "first_compatible_taxon_id": first_tid,
        "coarse_start": coarse_start,
        "n_identifications": len(ids),
        "n_compatible_identifications": len(compatible),
    })

    if coarse_start:
        eligible.append((o, compatible))

coarse_audit_df = pd.DataFrame(audit_rows)
coarse_audit_df.to_csv(
    OUTDIR/"coarse_start_screening.csv",
    index=False
)

print("Research-grade species observations screened:", len(coarse_audit_df))
print("Genuinely coarse-start:", len(eligible))

if len(eligible) < 500:
    raise RuntimeError(
        f"Only {len(eligible)} genuinely coarse-start observations remain. "
        "Increase TARGET_CANDIDATE_OBSERVATIONS or broaden the date window."
    )


## 6. Select a supported induced genus→species hierarchy

In [ ]:

support_rows = []

for o, compatible in eligible:
    sid = int(o["taxon"]["id"])
    gid = rank_ancestor_id(o, "genus")

    support_rows.append({
        "observation_id": int(o["id"]),
        "species_id": sid,
        "species_name": taxa.get(sid, o["taxon"]).get("name"),
        "genus_id": gid,
        "genus_name": taxa.get(gid, {}).get("name"),
    })

support_df = pd.DataFrame(support_rows)

species_counts = (
    support_df.groupby(
        ["genus_id","genus_name","species_id","species_name"]
    )
    .size()
    .reset_index(name="n")
)

eligible_species = species_counts[
    species_counts.n >= MIN_OBS_PER_SPECIES
]

genus_support = (
    eligible_species.groupby(["genus_id","genus_name"])
    .agg(
        eligible_species=("species_id","nunique"),
        eligible_observations=("n","sum"),
    )
    .reset_index()
)

selected_genera = (
    genus_support[
        genus_support.eligible_species >= MIN_SPECIES_PER_GENUS
    ]
    .sort_values("eligible_observations", ascending=False)
    .head(MAX_GENERA)
)

if len(selected_genera) < 3:
    raise RuntimeError(
        "Fewer than three genera survive support thresholds. "
        "Increase candidate pool or lower MIN_OBS_PER_SPECIES."
    )

selected_species_parts = []

for gid in selected_genera.genus_id:
    sub = (
        eligible_species[
            eligible_species.genus_id == gid
        ]
        .sort_values("n", ascending=False)
        .head(MAX_SPECIES_PER_GENUS)
    )
    selected_species_parts.append(sub)

selected_species_df = pd.concat(
    selected_species_parts,
    ignore_index=True
)

SELECTED_SPECIES = set(
    selected_species_df.species_id.astype(int)
)

selected_species_df.to_csv(
    OUTDIR/"selected_species.csv",
    index=False
)

print(selected_species_df.to_string(index=False))


## 7. Reconstruct trajectories relative to the first compatible coarse ID

In [ ]:

def model_state(tid, sid, gid, lineage):
    if tid == sid:
        return f"S:{sid}", 2

    if gid in lineage:
        gidx = lineage.index(gid)

        if tid in lineage and lineage.index(tid) >= gidx:
            return f"G:{gid}", 1

    return "ROOT", 0

trajectory_rows = []
event_rows = []

for o, compatible in eligible:
    sid = int(o["taxon"]["id"])

    if sid not in SELECTED_SPECIES:
        continue

    gid = rank_ancestor_id(o, "genus")
    lineage = final_lineage(o)
    lineage_set = set(lineage)

    t0 = compatible[0][0]
    deepest = ("ROOT",0)
    states = []

    for d in HORIZON_DAYS:
        cutoff = t0 + pd.Timedelta(days=d)
        deepest = ("ROOT",0)

        for t, tid, rank, ident in compatible:
            if t > cutoff:
                break

            state, depth = model_state(
                tid, sid, gid, lineage
            )

            if depth > deepest[1]:
                deepest = (state,depth)

        states.append(deepest[0])

    all_ids = list(o.get("identifications") or [])
    incompatible = 0
    withdrawn = 0

    for ident in all_ids:
        itax = ident.get("taxon") or {}

        if itax.get("id") is None:
            continue

        tid = int(itax["id"])
        comp = tid in lineage_set

        if not comp:
            incompatible += 1

        if ident.get("current") is False:
            withdrawn += 1

        event_rows.append({
            "observation_id": int(o["id"]),
            "identification_id": ident.get("id"),
            "created_at": ident.get("created_at"),
            "taxon_id": tid,
            "rank": itax.get("rank"),
            "compatible": comp,
            "current": ident.get("current"),
        })

    trajectory_rows.append({
        "observation_id": int(o["id"]),
        "species_id": sid,
        "species_name": taxa.get(sid, o["taxon"]).get("name"),
        "genus_id": gid,
        "genus_name": taxa.get(gid, {}).get("name"),
        "first_compatible_time": t0,
        "n_identifications": len(all_ids),
        "n_incompatible_identifications": incompatible,
        "n_withdrawn_identifications": withdrawn,
        **{
            f"state_{d}d": state
            for d,state in zip(HORIZON_DAYS, states)
        }
    })

traj_df = pd.DataFrame(trajectory_rows)
events_df = pd.DataFrame(event_rows)

traj_df.to_csv(
    OUTDIR/"reconstructed_coarse_start_trajectories.csv",
    index=False
)
events_df.to_csv(
    OUTDIR/"identification_event_audit.csv",
    index=False
)

print("Selected coarse-start trajectories:", len(traj_df))

for d in HORIZON_DAYS:
    col = f"state_{d}d"
    species_frac = traj_df[col].astype(str).str.startswith("S:").mean()
    genus_frac = traj_df[col].astype(str).str.startswith("G:").mean()
    root_frac = (traj_df[col] == "ROOT").mean()

    print(
        f"{d:>3}d: species={species_frac:.3f}, "
        f"genus={genus_frac:.3f}, root={root_frac:.3f}"
    )


## 8. Build hierarchy and empirical crossing masses

In [ ]:

species_to_genus = (
    traj_df[["species_id","genus_id"]]
    .drop_duplicates()
    .set_index("species_id")
    .genus_id.astype(int)
    .to_dict()
)

species_ids = sorted(traj_df.species_id.astype(int).unique())
genus_ids = sorted(traj_df.genus_id.astype(int).unique())

leaf_index = {
    sid:i for i,sid in enumerate(species_ids)
}
nL = len(species_ids)

children = {
    "ROOT": [f"G:{gid}" for gid in genus_ids]
}

for gid in genus_ids:
    children[f"G:{gid}"] = [
        f"S:{sid}"
        for sid in species_ids
        if species_to_genus[sid] == gid
    ]

def subtree_species(node):
    if node.startswith("S:"):
        return [int(node.split(":")[1])]

    if node.startswith("G:"):
        gid = int(node.split(":")[1])
        return [
            sid for sid in species_ids
            if species_to_genus[sid] == gid
        ]

    return list(species_ids)

def subtree_vector(node):
    v = np.zeros(nL)

    for sid in subtree_species(node):
        v[leaf_index[sid]] = 1.0

    return v

def state_in_subtree(state, child):
    state = str(state)

    if child.startswith("G:"):
        gid = int(child.split(":")[1])

        if state == child:
            return True

        if state.startswith("S:"):
            sid = int(state.split(":")[1])
            return species_to_genus.get(sid) == gid

        return False

    if child.startswith("S:"):
        return state == child

    return state == "ROOT"

N = len(traj_df)
FINAL_H = HORIZON_DAYS[-1]
r = defaultdict(lambda: defaultdict(dict))

for u,chs in children.items():
    for c in chs:
        for d in HORIZON_DAYS:
            r[u][c][d] = float(
                np.mean([
                    state_in_subtree(s,c)
                    for s in traj_df[f"state_{d}d"]
                ])
            )

p_eventual = np.zeros(nL)

for sid,count in traj_df.species_id.value_counts().items():
    p_eventual[leaf_index[int(sid)]] = count/N

crossing_rows = []

for u,chs in children.items():
    for c in chs:
        for d in HORIZON_DAYS:
            crossing_rows.append({
                "parent":u,
                "child":c,
                "horizon_days":d,
                "crossing_mass":r[u][c][d],
                "eventual_subtree_mass":float(
                    subtree_vector(c) @ p_eventual
                ),
            })

pd.DataFrame(crossing_rows).to_csv(
    OUTDIR/"crossing_masses.csv",
    index=False
)


## 9. Sample-analogue sensitivity polytope

In [ ]:

Aeq = np.ones((1,nL))
beq = np.array([1.0])
bounds = [(0,None)]*nL

def build_constraints(gamma=None, horizons=None, eta=0.0):
    if horizons is None:
        horizons = HORIZON_DAYS

    A = []
    b = []

    # Final-horizon candidate compatibility.
    for u,chs in children.items():
        for c in chs:
            A.append(-subtree_vector(c))
            b.append(-r[u][c][FINAL_H])

    if gamma is not None:
        for u,chs in children.items():
            vv = {c:subtree_vector(c) for c in chs}

            for d in horizons:
                for j in chs:
                    for k in chs:
                        if j == k:
                            continue

                        rj = r[u][j][d]
                        rk = r[u][k][d]

                        A.append(
                            rj*vv[k]
                            - gamma*rk*vv[j]
                        )
                        b.append(
                            (1-gamma)*rj*rk + eta
                        )

    return np.asarray(A), np.asarray(b)

def solve_bounds(A,b):
    lows = []
    highs = []

    for k in range(nL):
        c = np.zeros(nL)
        c[k] = 1.0

        lo = linprog(
            c,
            A_ub=A,
            b_ub=b,
            A_eq=Aeq,
            b_eq=beq,
            bounds=bounds,
            method="highs"
        )
        hi = linprog(
            -c,
            A_ub=A,
            b_ub=b,
            A_eq=Aeq,
            b_eq=beq,
            bounds=bounds,
            method="highs"
        )

        if not (lo.success and hi.success):
            return None

        lows.append(lo.fun)
        highs.append(-hi.fun)

    return np.asarray(lows), np.asarray(highs)

def minimum_relaxation(gamma, horizons):
    Acompat,bcompat = build_constraints(
        gamma=None,
        horizons=horizons,
        eta=0.0
    )

    Aall,ball = build_constraints(
        gamma=gamma,
        horizons=horizons,
        eta=0.0
    )

    ncompat = len(Acompat)
    As = Aall[ncompat:]
    bs = ball[ncompat:]

    Aaug = []
    baug = []

    for row,rhs in zip(Acompat,bcompat):
        Aaug.append(np.r_[row,0.0])
        baug.append(rhs)

    for row,rhs in zip(As,bs):
        Aaug.append(np.r_[row,-1.0])
        baug.append(rhs)

    c = np.zeros(nL+1)
    c[-1] = 1.0

    res = linprog(
        c,
        A_ub=np.asarray(Aaug),
        b_ub=np.asarray(baug),
        A_eq=np.r_[np.ones(nL),0.0][None,:],
        b_eq=[1.0],
        bounds=[(0,None)]*(nL+1),
        method="highs"
    )

    if not res.success:
        return np.nan

    return float(res.x[-1])

A_cand,b_cand = build_constraints(gamma=None)
cand_bounds = solve_bounds(A_cand,b_cand)

if cand_bounds is None:
    raise RuntimeError("Candidate-only polytope infeasible")

candidate_mean_width = float(
    np.mean(cand_bounds[1]-cand_bounds[0])
)

print("Candidate-only mean width:", candidate_mean_width)


## 10. Gamma profile

In [ ]:

profile_rows = []
bounds_parts = []

for gamma in GAMMA_GRID:
    eta_all = minimum_relaxation(
        gamma,
        HORIZON_DAYS
    )
    eta_final = minimum_relaxation(
        gamma,
        [FINAL_H]
    )

    A_all,b_all = build_constraints(
        gamma=gamma,
        horizons=HORIZON_DAYS,
        eta=eta_all + 1e-12
    )
    A_final,b_final = build_constraints(
        gamma=gamma,
        horizons=[FINAL_H],
        eta=eta_final + 1e-12
    )

    all_bounds = solve_bounds(A_all,b_all)
    final_bounds = solve_bounds(A_final,b_final)

    if all_bounds is None or final_bounds is None:
        continue

    all_width = all_bounds[1]-all_bounds[0]
    final_width = final_bounds[1]-final_bounds[0]

    profile_rows.append({
        "gamma":gamma,
        "eta_all_horizons":eta_all,
        "eta_final_horizon":eta_final,
        "candidate_mean_width":candidate_mean_width,
        "final_mean_width":float(final_width.mean()),
        "all_mean_width":float(all_width.mean()),
        "all_tightening_vs_candidate":float(
            1 - all_width.mean()/candidate_mean_width
        ),
        "all_tightening_vs_final":float(
            1 - all_width.mean()/final_width.mean()
            if final_width.mean() > 0 else 0
        ),
        "strictly_tightened_leaves_vs_final":int(
            np.sum(all_width < final_width-1e-8)
        ),
        # Retrospective diagnostic only; NOT a pass criterion.
        "eventual_species_proportions_contained":bool(
            np.all(
                (p_eventual >= all_bounds[0]-1e-9)
                & (p_eventual <= all_bounds[1]+1e-9)
            )
        ),
    })

    bounds_parts.append(pd.DataFrame({
        "species_id":species_ids,
        "species_name":[
            taxa.get(sid,{}).get("name",str(sid))
            for sid in species_ids
        ],
        "eventual_proportion":p_eventual,
        "lower":all_bounds[0],
        "upper":all_bounds[1],
        "width":all_width,
        "gamma":gamma,
    }))

profile_df = pd.DataFrame(profile_rows)
bounds_df = pd.concat(bounds_parts, ignore_index=True)

profile_df.to_csv(
    OUTDIR/"gamma_profile.csv",
    index=False
)
bounds_df.to_csv(
    OUTDIR/"leaf_bounds_by_gamma.csv",
    index=False
)

print(profile_df.to_string(index=False))


## 11. Real-data gate

In [ ]:

# Use sample-analogue criteria only.
usable = profile_df[
    (profile_df.eta_all_horizons <= 1e-3)
    & (profile_df.all_tightening_vs_candidate >= 0.05)
].copy()

gamma_star = (
    float(usable.sort_values("gamma").iloc[0]["gamma"])
    if len(usable)
    else None
)

species_fraction = {
    d: float(
        traj_df[f"state_{d}d"]
        .astype(str)
        .str.startswith("S:")
        .mean()
    )
    for d in HORIZON_DAYS
}

compatible_event_rate = float(
    events_df.compatible.mean()
)

if gamma_star is not None:
    star = profile_df[
        profile_df.gamma == gamma_star
    ].iloc[0]

    temporal_ok = bool(
        star.all_tightening_vs_final >= 0.005
        or star.strictly_tightened_leaves_vs_final >= 2
    )
else:
    temporal_ok = False

checks = {
    "at_least_500_selected_coarse_start_observations": bool(
        len(traj_df) >= 500
    ),
    "at_least_3_genera": bool(
        traj_df.genus_id.nunique() >= 3
    ),
    "at_least_8_species": bool(
        traj_df.species_id.nunique() >= 8
    ),
    "time_zero_is_genuinely_coarse": bool(
        species_fraction[0] == 0.0
    ),
    "meaningful_refinement_occurs_by_90d": bool(
        species_fraction[90] >= 0.30
    ),
    "not_everything_is_resolved_by_90d": bool(
        species_fraction[90] <= 0.95
    ),
    "compatible_identification_rate_at_least_90pct": bool(
        compatible_event_rate >= 0.90
    ),
    "usable_gamma_exists": bool(
        gamma_star is not None
    ),
    "multiple_horizons_add_information": bool(
        temporal_ok
    ),
}

verdict = (
    "PASS_TO_REAL_LEARNING_PILOT"
    if all(checks.values())
    else "STOP_AND_DIAGNOSE_COARSE_START_TRAJECTORIES"
)

summary = {
    "stage":"inaturalist_coarse_start_real_trajectory_pilot",
    "target_taxon_id":TARGET_TAXON_ID,
    "candidate_observations":len(candidate_obs_ids),
    "research_grade_species_screened":len(coarse_audit_df),
    "genuinely_coarse_start_before_support_filter":len(eligible),
    "selected_coarse_start_observations":len(traj_df),
    "selected_genera":int(traj_df.genus_id.nunique()),
    "selected_species":int(traj_df.species_id.nunique()),
    "species_fraction_by_horizon":species_fraction,
    "compatible_identification_rate":compatible_event_rate,
    "candidate_mean_width":candidate_mean_width,
    "gamma_star":gamma_star,
    "checks":checks,
    "verdict":verdict,
    "interpretation":(
        "This pilot explicitly targets observations whose first compatible "
        "identification is coarse. Eventual research-grade species labels are "
        "used only for lineage reconstruction and retrospective diagnostics. "
        "Finite-sample pass/fail follows the sample-analogue estimator, not "
        "containment of the realized hidden species proportions."
    )
}

with open(
    OUTDIR/"coarse_start_real_trajectory_pilot_summary.json",
    "w"
) as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2))
print("\nVERDICT:", verdict)


## 12. Plots and archive

In [ ]:

plt.figure(figsize=(7.2,4.8))
plt.plot(
    list(species_fraction.keys()),
    list(species_fraction.values()),
    marker="o"
)
plt.xlabel("Days since first compatible coarse identification")
plt.ylabel("Fraction at species level")
plt.title("Observed refinement trajectory")
plt.tight_layout()
plt.savefig(
    OUTDIR/"species_refinement_curve.png",
    dpi=160
)
plt.close()

plt.figure(figsize=(7.2,4.8))
plt.plot(
    profile_df.gamma,
    profile_df.all_tightening_vs_candidate,
    marker="o",
    label="All horizons vs candidate-only"
)
plt.plot(
    profile_df.gamma,
    profile_df.all_tightening_vs_final,
    marker="s",
    label="All horizons vs final horizon"
)
plt.axhline(0, linestyle="--")
plt.xlabel("Gamma")
plt.ylabel("Relative interval-width reduction")
plt.title("Coarse-start real-data identification gain")
plt.legend()
plt.tight_layout()
plt.savefig(
    OUTDIR/"gamma_identification_gain.png",
    dpi=160
)
plt.close()

zip_name = "inaturalist_coarse_start_trajectory_pilot_results.zip"

with zipfile.ZipFile(
    zip_name,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for p in OUTDIR.iterdir():
        zf.write(p, arcname=p.name)

print("Results archive:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    pass
